In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau, StepLR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

In [3]:
np.random.seed(42)
torch.manual_seed(42)

In [4]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data"
columns = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class']
df = pd.read_csv(url, names=columns)

In [5]:
print(df.info())
print('-'*40)
print(' ')
print(df.head())
print(' ')
print('-'*40)
print(' ')
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1728 entries, 0 to 1727
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   buying    1728 non-null   object
 1   maint     1728 non-null   object
 2   doors     1728 non-null   object
 3   persons   1728 non-null   object
 4   lug_boot  1728 non-null   object
 5   safety    1728 non-null   object
 6   class     1728 non-null   object
dtypes: object(7)
memory usage: 94.6+ KB
None
----------------------------------------
 
  buying  maint doors persons lug_boot safety  class
0  vhigh  vhigh     2       2    small    low  unacc
1  vhigh  vhigh     2       2    small    med  unacc
2  vhigh  vhigh     2       2    small   high  unacc
3  vhigh  vhigh     2       2      med    low  unacc
4  vhigh  vhigh     2       2      med    med  unacc
 
----------------------------------------
 
       buying  maint doors persons lug_boot safety  class
count    1728   1728  1728    1728     172

In [6]:
df['class'].value_counts()

,count
class,
unacc,1210
acc,384
good,69
vgood,65


In [7]:
X = df.drop('class', axis=1)
y = df['class']

In [8]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [9]:
class_names = le.classes_
num_classes = len(class_names)

In [10]:
X_train,X_test,y_train,y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42,stratify=y_encoded
)

In [11]:
buying_order = ['vhigh','high','med','low']
maint_order = ['vhigh', 'high', 'med', 'low']
doors_order = ['2', '3', '4', '5more']  # 5more > 4
persons_order = ['2', '4', 'more']
lug_boot_order = ['small', 'med', 'big']
safety_order = ['low', 'med', 'high']
ordinal_category = [buying_order,maint_order,doors_order,persons_order,lug_boot_order,safety_order]


In [12]:
data_version = {}
ordinal_enc = OrdinalEncoder(categories=ordinal_category)
X_train_ord = ordinal_enc.fit_transform(X_train)
X_test_ord = ordinal_enc.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['StandardScaler'] = (X_train_scaled, X_test_scaled)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['MinMaxScaler'] = (X_train_scaled, X_test_scaled)

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_ord)
X_test_scaled = scaler.transform(X_test_ord)
data_version['RobustScaler'] = (X_train_scaled, X_test_scaled)

ordinal_ohe = OneHotEncoder(drop='first',sparse_output=False)
X_train_ohe = ordinal_ohe.fit_transform(X_train)
X_test_ohe = ordinal_ohe.transform(X_test)
data_version['ohe'] =(X_train_ohe, X_test_ohe)

scaler = StandardScaler()
X_train_ohe_scaled = scaler.fit_transform(X_train_ohe)
X_test_ohe_scaled = scaler.transform(X_test_ohe)
data_version['StandardScaler_ohe'] = (X_train_ohe_scaled, X_test_ohe_scaled)

for k in data_version:
    print(f'{k} : {data_version[k][0].shape}')


StandardScaler : (1382, 6)
MinMaxScaler : (1382, 6)
RobustScaler : (1382, 6)
ohe : (1382, 15)
StandardScaler_ohe : (1382, 15)


In [13]:
def create_dataloaders(X_train, X_test, y_train, y_test,batch_size):
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    train_dataset = TensorDataset(X_train_t, y_train_t)
    test_dataset = TensorDataset(X_test_t, y_test_t)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader


In [14]:
loaders = {}
for name, (Xtr,xts) in data_version.items():
    train_loader, test_loader = create_dataloaders(Xtr, xts, y_train, y_test,batch_size=32)
    loaders[name] = (train_loader, test_loader)

In [15]:
class FlexibleNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=[64,32], output_dim=4,dropout=0.3,
                 use_batchnorm=True,activation='relu'):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dim:
            layers.append(nn.Linear(prev_dim, h_dim))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h_dim))
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
            elif activation == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.1))
            else:
                raise ValueError("Unsupported activation")
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self,x):
        return self.net(x)

In [16]:
def train_model(model,train_loader,test_loader,criterion,
                optimizer,scheduler=None,epochs=100,patience=10,device='cpu',verbose=True):
  model.to(device)
  train_losses, test_losses = [],[]
  train_accs, test_accs = [],[]

  best_test_acc = 0
  best_model_state = None
  patience_counter = 0
  for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
      inputs, labels = inputs.to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = model(inputs)
      loss = criterion(outputs,labels)
      loss.backward()
      optimizer.step()
      train_loss += loss.item() * inputs.size(0)
      _, predicted = torch.max(outputs, 1)
      total_train += labels.size(0)
      correct_train += (predicted == labels).sum().item()

    avg_train_loss = train_loss / total_train
    train_acc = 100 * correct_train / total_train
    train_losses.append(avg_train_loss)
    train_accs.append(train_acc)

    model.eval()
    test_loss = 0.0
    correct_test = 0
    total_test = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
      for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs,labels)
        test_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_test += labels.size(0)
        correct_test += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

      avg_test_loss = test_loss / total_test
      test_acc = 100 * correct_test / total_test
      test_losses.append(avg_test_loss)
      test_accs.append(test_acc)

      if scheduler:
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(avg_test_loss)
        else:
            scheduler.step()

      if test_acc > best_test_acc:
          best_test_acc = test_acc
          best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
          patience_counter = 0
      else:
          patience_counter += 1
          if patience_counter >= patience:
              if verbose:
                  print(f"Early stopping at epoch {epoch+1}")
                  break

      if verbose and (epoch+1) % 20 == 0:
          print(f"Epoch {epoch+1:3d} | Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

  model.load_state_dict(best_model_state)
  return model, train_losses, test_losses, train_accs, test_accs, all_preds, all_labels


In [17]:
EPOCHS = 150
BATCH_SIZE = 32
PATIENCE = 15

In [18]:
results_encoding = {}
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for name, (train_loader, test_loader) in loaders.items():
    print(f"\n=== Тестирование: {name} ===")
    input_dim = data_version[name][0].shape[1]
    model = FlexibleNet(input_dim, hidden_dim=[64, 32], output_dim=num_classes,
                        dropout=0.5, use_batchnorm=True, activation='relu')
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    model, *history, preds, labels = train_model(
        model, train_loader, test_loader, criterion, optimizer, scheduler,
        epochs=EPOCHS, patience=PATIENCE, device=device, verbose=False
    )
    final_acc = max(history[3])
    results_encoding[name] = final_acc
    print(f"Final Test Accuracy: {final_acc:.2f}%")


=== Тестирование: StandardScaler ===
Final Test Accuracy: 97.40%

=== Тестирование: MinMaxScaler ===
Final Test Accuracy: 97.40%

=== Тестирование: RobustScaler ===
Final Test Accuracy: 97.40%

=== Тестирование: ohe ===
Final Test Accuracy: 97.98%

=== Тестирование: StandardScaler_ohe ===
Final Test Accuracy: 98.84%
